In [1]:
import heapq

In [2]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost      # g(n): tổng chi phí từ gốc đến node này

    def __lt__(self, other):            # để heapq so sánh các node theo g(n)
        return self.path_cost < other.path_cost

In [3]:
def is_goal(state, goal_condition):
    if callable(goal_condition):
        return goal_condition(state)
    return state == goal_condition

In [4]:
def PriorityQueue():
    queue = []
    return queue

In [5]:
def enqueue(queue, node):
    heapq.heappush(queue, node)

In [6]:
def dequeue(queue):
    if is_empty(queue):
        return None
    return heapq.heappop(queue)         # node có g(n) nhỏ nhất

In [7]:
def is_empty(queue):
    return len(queue) == 0

In [8]:
def child_node(problem, parent, action):
    new_state = problem.result(parent.state, action)
    cost = parent.path_cost + problem.action_cost(parent.state, action, new_state)
    child = Node(
        state=new_state,
        parent=parent,
        action=action,
        path_cost=cost
    )
    return child

In [9]:
def solution(node):
    path = []
    actions = []
    curr = node
    while curr is not None:
        path.append(curr.state)
        if curr.parent is not None:     # node gốc không có hành động
            actions.append(curr.action)
        curr = curr.parent
    path.reverse()
    actions.reverse()
    return path, actions

In [10]:
class Problem:
    def __init__(self, initial_state, goal_condition, actions, result, action_cost=None):
        self.initial_state = initial_state
        self.goal_condition = goal_condition
        self._actions_fn = actions
        self._result_fn = result
        self._action_cost_fn = action_cost

    def is_goal(self, state):
        return is_goal(state, self.goal_condition)

    def actions(self, state):
        return self._actions_fn(state)

    def result(self, state, action):
        return self._result_fn(state, action)

    def action_cost(self, state, action, new_state):
        if self._action_cost_fn is None:
            return 1                    # mặc định mỗi hành động có chi phí 1
        return self._action_cost_fn(state, action, new_state)   # phải không âm

In [11]:
def find_in_frontier(frontier, state):
    for i, node in enumerate(frontier):
        if node.state == state:
            return i
    return None

def UCS(problem):
    node = Node(problem.initial_state, path_cost=0)
    frontier = PriorityQueue()
    enqueue(frontier, node)
    explored = set()

    while not is_empty(frontier):
        node = dequeue(frontier)
        if problem.is_goal(node.state):
            path, actions = solution(node)
            return {
                "path": path,
                "actions": actions,
                "cost": node.path_cost,
                "goal_node": node
            }

        explored.add(node.state)
        for action in problem.actions(node.state):
            child = child_node(problem, node, action)
            index = find_in_frontier(frontier, child.state)
            if child.state not in explored and index is None:
                enqueue(frontier, child)
            elif index is not None and frontier[index].path_cost > child.path_cost:
                frontier[index] = child
                heapq.heapify(frontier)

    return None

In [ ]:
initial_state = (
    (1, 2, 3),
    (4, 0, 6),
    (7, 5, 8)
)

goal_state = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c

def get_actions_8puzzle(state):
    r, c = find_blank(state)
    actions = []
    if r > 0:
        actions.append('Up')
    if r < 2:
        actions.append('Down')
    if c > 0:
        actions.append('Left')
    if c < 2:
        actions.append('Right')

    return actions

def get_result_8puzzle(state, action):
    r, c = find_blank(state)
    grid = [list(row) for row in state]
    if action == 'Up':
        grid[r][c], grid[r - 1][c] = grid[r - 1][c], grid[r][c]
    elif action == 'Down':
        grid[r][c], grid[r + 1][c] = grid[r + 1][c], grid[r][c]
    elif action == 'Left':
        grid[r][c], grid[r][c - 1] = grid[r][c - 1], grid[r][c]
    elif action == 'Right':
        grid[r][c], grid[r][c + 1] = grid[r][c + 1], grid[r][c]
    return tuple(tuple(row) for row in grid)

# Chi phí mỗi hành động: đi dọc (Up, Down) = 1, đi ngang (Left, Right) = 2
def get_cost_8puzzle(state, action, new_state):
    if action in ('Up', 'Down'):
        return 1
    return 2

def print_grid(state):
    for row in state:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))

problem = Problem(
    initial_state=initial_state,
    goal_condition=goal_state,
    actions=get_actions_8puzzle,
    result=get_result_8puzzle,
    action_cost=get_cost_8puzzle       
)

print("Trạng thái ban đầu:")
print_grid(initial_state)
print("\n Trạng thái đích:")
print_grid(goal_state)

print("\n Đang chạy thuật toán UCS...")
result = UCS(problem)

if result:
    actions = result["actions"]
    path = result["path"]

    print(f"\n Tìm thấy đường đi sau {len(actions)} bước, tổng chi phí = {result['cost']}")
    print(f"Chuỗi hành động: {' -> '.join(actions)}")

    g = 0
    for i, state in enumerate(path):
        if i == 0:
            print(f"Bước 0 (Trạng thái bắt đầu, g = 0):")
        else:
            cost = problem.action_cost(path[i - 1], actions[i - 1], state)
            g += cost
            print(f"Bước {i} (Sau khi ô 0 đi: {actions[i - 1]}, chi phí +{cost}, g = {g}):")
        print_grid(state)
        print()
else:
    print("\n Không tìm thấy lời giải!")

Trạng thái ban đầu:
  1 2 3
  4 _ 6
  7 5 8

 Trạng thái đích:
  1 2 3
  4 5 6
  7 8 _

 Đang chạy thuật toán UCS...

 Tìm thấy đường đi sau 2 bước, tổng chi phí = 3
Chuỗi hành động: Down -> Right
Bước 0 (Trạng thái bắt đầu, g = 0):
  1 2 3
  4 _ 6
  7 5 8

Bước 1 (Sau khi ô 0 đi: Down, chi phí +1, g = 1):
  1 2 3
  4 5 6
  7 _ 8

Bước 2 (Sau khi ô 0 đi: Right, chi phí +2, g = 3):
  1 2 3
  4 5 6
  7 8 _



In [13]:
initial_state = ('A', 'D', 'D')

def is_goal_vacuum(state):
    return state in [('A', 'C', 'C'), ('B', 'C', 'C')]

def get_actions_vacuum(state):
    return ['Suck', 'Left', 'Right']

def get_result_vacuum(state, action):
    pos, status_A, status_B = state

    if action == 'Suck':
        if pos == 'A':
            return ('A', 'C', status_B)
        elif pos == 'B':
            return ('B', status_A, 'C')

    elif action == 'Left':
        return ('A', status_A, status_B)

    elif action == 'Right':
        return ('B', status_A, status_B)

    return state

def get_cost_vacuum(state, action, new_state):
    if action == 'Suck':
        return 1
    return 2

def print_state(state):
    pos, status_A, status_B = state
    desc_A = "Clean (C)" if status_A == 'C' else "Dirty (D)"
    desc_B = "Clean (C)" if status_B == 'C' else "Dirty (D)"
    print(f"  Robot đang ở phòng: [{pos}] | Phòng A: {desc_A:<9} | Phòng B: {desc_B:<9}")

problem = Problem(
    initial_state=initial_state,
    goal_condition=is_goal_vacuum,
    actions=get_actions_vacuum,
    result=get_result_vacuum,
    action_cost=get_cost_vacuum
)

print("Trạng thái ban đầu:")
print_state(initial_state)
print("\n Trạng thái đích cần đạt: ('A', 'C', 'C') hoặc ('B', 'C', 'C')")

print("\n Đang chạy thuật toán UCS...")
result = UCS(problem)

if result:
    actions = result["actions"]
    path = result["path"]

    print(f"\n Tìm thấy lời giải sau {len(actions)} bước, tổng chi phí = {result['cost']}")
    print(f"Chuỗi hành động: {' -> '.join(actions)}")

    g = 0
    for i, state in enumerate(path):
        if i == 0:
            print(f"Bước 0 (Trạng thái bắt đầu, g = 0):")
        else:
            cost = problem.action_cost(path[i - 1], actions[i - 1], state)
            g += cost
            print(f"Bước {i} (Hành động: {actions[i - 1]}, chi phí +{cost}, g = {g}):")
        print_state(state)
        print()
else:
    print("\n Không tìm thấy lời giải!")

Trạng thái ban đầu:
  Robot đang ở phòng: [A] | Phòng A: Dirty (D) | Phòng B: Dirty (D)

 Trạng thái đích cần đạt: ('A', 'C', 'C') hoặc ('B', 'C', 'C')

 Đang chạy thuật toán UCS...

 Tìm thấy lời giải sau 3 bước, tổng chi phí = 4
Chuỗi hành động: Suck -> Right -> Suck
Bước 0 (Trạng thái bắt đầu, g = 0):
  Robot đang ở phòng: [A] | Phòng A: Dirty (D) | Phòng B: Dirty (D)

Bước 1 (Hành động: Suck, chi phí +1, g = 1):
  Robot đang ở phòng: [A] | Phòng A: Clean (C) | Phòng B: Dirty (D)

Bước 2 (Hành động: Right, chi phí +2, g = 3):
  Robot đang ở phòng: [B] | Phòng A: Clean (C) | Phòng B: Dirty (D)

Bước 3 (Hành động: Suck, chi phí +1, g = 4):
  Robot đang ở phòng: [B] | Phòng A: Clean (C) | Phòng B: Clean (C)

